# FitNova GPU Training

**Before cell 1:** Menu bar → Runtime → Change runtime type → T4 GPU → Save.

Run every cell top to bottom. Do NOT skip cells.

---
### ⚡ If the runtime disconnected mid-run
Re-run **cells 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10** in order.  
Cell 6 checks your Drive and **skips every video that was already extracted** — it picks up where it left off automatically.

## Cell 1 — Check GPU

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
assert gpus, 'NO GPU! Menu > Runtime > Change runtime type > T4 GPU > Save, then re-run.'
print('TF:', tf.__version__, '| GPUs:', gpus)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## Cell 2 — Mount Google Drive

A popup will ask for permission. Click through and allow.

**Before running:** you must have uploaded `videos.zip` to the root of your Google Drive (`MyDrive/videos.zip`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
src = '/content/drive/MyDrive/videos.zip'
assert os.path.exists(src), f'videos.zip not found at {src} — did you upload it to Drive root?'
size_gb = os.path.getsize(src) / 1e9
print(f'Found videos.zip: {size_gb:.2f} GB')

## Cell 3 — Extract videos (OPTIONAL, skip unless you also want to run visualize_pipeline on Colab)

The Fit3D training path does NOT need videos — it reads the MoCap JSONs directly.
You can skip this cell. Run it only if you want mp4s on Colab for later inspection.


In [ ]:
import shutil, zipfile, os

target = '/content/videos'
n_mp4 = sum(1 for _, _, fs in os.walk(target) for f in fs if f.endswith('.mp4'))
if n_mp4 >= 100:
    print(f'Already extracted ({n_mp4} .mp4 files). Skipping.')
elif not os.path.exists('/content/drive/MyDrive/videos.zip'):
    print('videos.zip not on Drive — skipping (not required for Fit3D training).')
else:
    print('Copying videos.zip from Drive to local disk...')
    shutil.copy('/content/drive/MyDrive/videos.zip', '/content/videos.zip')
    print('Extracting...')
    with zipfile.ZipFile('/content/videos.zip') as zf:
        zf.extractall('/content/videos')
    os.remove('/content/videos.zip')
    count = sum(1 for _, _, fs in os.walk('/content/videos')
                for f in fs if f.endswith('.mp4'))
    print(f'Extracted {count} .mp4 files')


## Cell 4 — Load source code (crash-safe)

**First time:** upload `fitnova_src.zip` via the file picker.
It is also saved to `MyDrive/fitnova_src.zip` automatically.

**After reconnect:** the cell loads from Drive automatically. No re-upload needed.

If you rebuild the zip on your laptop (e.g. after editing any backend file),
**delete `MyDrive/fitnova_src.zip` first**, otherwise this cell will restore the old one.


In [ ]:
import os, shutil, zipfile

DRIVE_SRC = '/content/drive/MyDrive/fitnova_src.zip'
LOCAL_SRC  = '/content/fitnova_src.zip'

if os.path.exists(DRIVE_SRC):
    print('Loading fitnova_src.zip from Drive (no upload needed)...')
    shutil.copy(DRIVE_SRC, LOCAL_SRC)
    src_zip = LOCAL_SRC
else:
    print('fitnova_src.zip not on Drive yet -- use file picker below.')
    from google.colab import files
    uploaded = files.upload()
    src_zip = list(uploaded.keys())[0]
    shutil.copy(src_zip, DRIVE_SRC)
    print('Saved to Drive: MyDrive/fitnova_src.zip')

# Clean any previous extraction so edits in the new zip take effect
if os.path.exists('/content/backend'):
    shutil.rmtree('/content/backend')
with zipfile.ZipFile(src_zip) as zf:
    zf.extractall('/content')
print('Source extracted')
!ls /content/backend/training


## Cell 5 — Install deps + set path


In [ ]:
!pip install -q mediapipe scikit-learn h5py tqdm
import sys, os
sys.path.insert(0, '/content')
os.chdir('/content')
print('Ready')


## Cell 6 — Extract Fit3D annotations (~10 sec, crash-safe)

Fit3D training reads `joints3d_25/*.json` + `rep_ann.json` files.
These are small (<50 MB total for all 8 subjects) and must be uploaded to
`MyDrive/fit3d_annotations.zip` once.

**To build the zip on your laptop:**
```
python _build_fit3d_annotations_zip.py
```
Then upload the resulting `fit3d_annotations.zip` to the root of your Google Drive.

- **First run**: extracts from Drive to `/content/fit3d_data/`.
- **After reconnect**: detects existing extraction and skips.


In [ ]:
import os, shutil, zipfile

FIT3D_DRIVE = '/content/drive/MyDrive/fit3d_annotations.zip'
FIT3D_LOCAL = '/content/fit3d_data'

if os.path.exists(FIT3D_LOCAL) and os.path.isdir(FIT3D_LOCAL):
    n_json = sum(1 for _, _, fs in os.walk(FIT3D_LOCAL)
                 for f in fs if f.endswith('.json'))
    if n_json >= 50:
        print(f'Fit3D annotations already extracted ({n_json} JSONs). Skipping.')
    else:
        shutil.rmtree(FIT3D_LOCAL)
        print('Incomplete previous extraction removed — re-extracting...')

if not os.path.exists(FIT3D_LOCAL):
    if not os.path.exists(FIT3D_DRIVE):
        raise FileNotFoundError(
            'fit3d_annotations.zip not found at MyDrive/fit3d_annotations.zip.\n'
            'Build it on your laptop:\n'
            '    python _build_fit3d_annotations_zip.py\n'
            'Then upload to Drive root.'
        )
    print('Copying fit3d_annotations.zip from Drive...')
    shutil.copy(FIT3D_DRIVE, '/content/fit3d_annotations.zip')
    print('Extracting...')
    with zipfile.ZipFile('/content/fit3d_annotations.zip') as zf:
        zf.extractall(FIT3D_LOCAL)
    os.remove('/content/fit3d_annotations.zip')

# Sanity check: expected subject folders
expected = ['s03', 's04', 's05', 's07', 's08', 's09', 's10', 's11']
missing = [s for s in expected if not os.path.isdir(os.path.join(FIT3D_LOCAL, s))]
if missing:
    raise RuntimeError(f'Missing subject folders under {FIT3D_LOCAL}: {missing}')

n_json = sum(1 for _, _, fs in os.walk(FIT3D_LOCAL)
             for f in fs if f.endswith('.json'))
print(f'Fit3D annotations ready: {n_json} JSONs across 8 subjects')
!ls /content/fit3d_data | head -20


## Cell 7 — Build training dataset from Fit3D MoCap (~3-5 min, crash-safe)

Builds `form_dataset_fit3d/` with:
- **Source**: Fit3D `joints3d_25` MoCap (clean 3D ground truth, view-invariant).
- **Sim-to-real**: random Y-axis rotation (±45°) + anisotropic Gaussian noise
  applied to training split only, to bridge MoCap → noisy MediaPipe at inference.
- **Split**: train = s03,s04,s05,s07,s08 · val = s09,s10 · **test = s11** (never touched).

- **First run**: builds and saves to `MyDrive/fitnova_dataset_fit3d/`.
- **After reconnect**: restores from Drive (~1 min). No rebuild needed.

To force a fresh build: delete `MyDrive/fitnova_dataset_fit3d/` from Drive, then rerun.


In [ ]:
import os, shutil

DATASET_LOCAL = '/content/backend/data/form_dataset_fit3d'
DATASET_DRIVE = '/content/drive/MyDrive/fitnova_dataset_fit3d'
FIT3D_ROOT    = '/content/fit3d_data'

if os.path.exists(os.path.join(DATASET_DRIVE, 'train', 'X_angles.npy')):
    # ── Fast restore path (~1-2 min) ──────────────────────────────────────
    print('Dataset found on Drive. Restoring to local disk...')
    if os.path.exists(DATASET_LOCAL):
        shutil.rmtree(DATASET_LOCAL)
    shutil.copytree(DATASET_DRIVE, DATASET_LOCAL)
    import numpy as np
    splits = {}
    for sp in ('train', 'val', 'test'):
        p = os.path.join(DATASET_LOCAL, sp, 'X_angles.npy')
        if os.path.exists(p):
            splits[sp] = len(np.load(p))
    print('  Splits:', splits)
    print('Restored from Drive. Skipping build.')
else:
    # ── Build path ────────────────────────────────────────────────────────
    assert os.path.exists(FIT3D_ROOT), \
        f'Fit3D root missing at {FIT3D_ROOT}. Run Cell 6 first.'
    print(f'Building dataset from Fit3D MoCap at {FIT3D_ROOT}...')
    print('  (Sim-to-real domain randomization applied to train split only)')
    !python -m backend.training.preprocessing.dataset_builder \
        {FIT3D_ROOT} \
        {DATASET_LOCAL} \
        --source fit3d

    if os.path.exists(os.path.join(DATASET_LOCAL, 'train', 'X_angles.npy')):
        import numpy as np
        splits = {}
        for sp in ('train', 'val', 'test'):
            p = os.path.join(DATASET_LOCAL, sp, 'X_angles.npy')
            if os.path.exists(p):
                splits[sp] = len(np.load(p))
        print('  Splits:', splits)
        print('Saving dataset to Drive for crash recovery (~2 min)...')
        if os.path.exists(DATASET_DRIVE):
            shutil.rmtree(DATASET_DRIVE)
        shutil.copytree(DATASET_LOCAL, DATASET_DRIVE)
        print('  Saved to MyDrive/fitnova_dataset_fit3d/')
    else:
        raise RuntimeError('Dataset not found after build -- check output above.')


## Cell 8 — SSL Skeleton Autoencoder Pretraining (~90 min on L4, crash-safe)

Trains a masked autoencoder on all subjects' data. No task labels used.
SSL encoder weights are saved to Drive immediately after training.

- **After reconnect**: detects saved encoder weights on Drive and skips training.
- **To force retrain**: delete `MyDrive/fitnova_results_fit3d/mt_tcn_encoder_ssl.weights.h5`.


In [ ]:
import os, shutil

SSL_LOCAL = '/content/backend/models/form_model/mt_tcn_encoder_ssl.weights.h5'
DRIVE_DIR = '/content/drive/MyDrive/fitnova_results_fit3d'
SSL_DRIVE = os.path.join(DRIVE_DIR, 'mt_tcn_encoder_ssl.weights.h5')

os.makedirs('/content/backend/models/form_model', exist_ok=True)
os.makedirs(DRIVE_DIR, exist_ok=True)

if os.path.exists(SSL_DRIVE):
    print('SSL encoder weights found on Drive. Restoring and skipping training.')
    shutil.copy(SSL_DRIVE, SSL_LOCAL)
    print(f'  Size: {os.path.getsize(SSL_LOCAL)/1e6:.1f} MB')
else:
    print('Running SSL pretraining (L4: ~90 min)...')
    !python -m backend.training.train_ssl_pretrain \
        --data-dir  /content/backend/data/form_dataset_fit3d \
        --model-dir /content/backend/models/form_model \
        --epochs 80 \
        --patience 15 \
        --batch-size 128
    if os.path.exists(SSL_LOCAL):
        shutil.copy(SSL_LOCAL, SSL_DRIVE)
        print(f'SSL weights saved to Drive: MyDrive/fitnova_results_fit3d/')
    else:
        raise RuntimeError('SSL weights not found after training -- check output above.')


## Cell 9 — Supervised fine-tune from SSL init (~60 min on L4, crash-safe)

Loads the SSL encoder from Cell 8 and fine-tunes all 5 task heads on the
Fit3D dataset. All model files are saved to Drive immediately after training.

- **After reconnect**: detects saved model on Drive and skips training.
- **To force retrain**: delete `MyDrive/fitnova_results_fit3d/mt_tcn_best.weights.h5`.


In [ ]:
import os, shutil

MODEL_DIR   = '/content/backend/models/form_model'
DRIVE_DIR   = '/content/drive/MyDrive/fitnova_results_fit3d'
MODEL_DRIVE = os.path.join(DRIVE_DIR, 'mt_tcn_best.weights.h5')
ALL_FILES   = [
    'mt_tcn_weights.weights.h5', 'mt_tcn_best.weights.h5',
    'model_config.json', 'angle_stats.npz', 'exercise_labels.json',
    'training_metrics.json', 'test_metrics.json', 'training_history.json',
]

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(DRIVE_DIR, exist_ok=True)

if os.path.exists(MODEL_DRIVE):
    print('Final model found on Drive. Restoring all files and skipping training.')
    for fn in ALL_FILES:
        src_f = os.path.join(DRIVE_DIR, fn)
        if os.path.exists(src_f):
            shutil.copy(src_f, os.path.join(MODEL_DIR, fn))
            print(f'  Restored {fn}')
else:
    print('Running supervised fine-tune from SSL init (L4: ~60 min)...')
    !python -m backend.training.train_form_model \
        --data_dir  /content/backend/data/form_dataset_fit3d \
        --model_dir /content/backend/models/form_model \
        --source    fit3d \
        --ssl-init  /content/backend/models/form_model/mt_tcn_encoder_ssl.weights.h5 \
        --batch-size 64
    # Save all model files to Drive immediately
    saved = []
    for fn in ALL_FILES:
        p = os.path.join(MODEL_DIR, fn)
        if os.path.exists(p):
            shutil.copy(p, os.path.join(DRIVE_DIR, fn))
            saved.append(fn)
    if saved:
        print(f'Saved to Drive: {saved}')
    else:
        raise RuntimeError('No model files found after training -- check output above.')


## Cell 10 — Verify + re-save results to Drive

Cells 8 and 9 already save to Drive immediately after training.
Run this cell at the end to do a final check and re-save anything missing.

After this cell prints `OK` for all files, go to `drive.google.com` and
download **all files** from `MyDrive/fitnova_results_fit3d/` to your laptop.

Place them all in: `backend/models/form_model/` on your laptop.


In [ ]:
import os, shutil

dst = '/content/drive/MyDrive/fitnova_results_fit3d'
os.makedirs(dst, exist_ok=True)
src_dir = '/content/backend/models/form_model'

print('=== Drive results check ===\n')
for fn in ['mt_tcn_weights.weights.h5', 'mt_tcn_best.weights.h5',
           'mt_tcn_encoder_ssl.weights.h5',
           'training_history.json', 'training_metrics.json',
           'test_metrics.json',
           'angle_stats.npz', 'exercise_labels.json', 'model_config.json']:
    p = os.path.join(src_dir, fn)
    if os.path.isfile(p):
        shutil.copy(p, dst)
        size = os.path.getsize(p) / 1e6
        print(f'  OK  {fn}  ({size:.1f} MB)')
    else:
        print(f'  MISSING  {fn}')

print()
print('Download ALL files from MyDrive/fitnova_results_fit3d/ to your laptop.')
print('Place them all in: backend/models/form_model/')
